# Lab 4.8 &mdash; Beyond Accuracy: Precision, Recall & Confusion on Real Predictions

**Level:** Intermediate &nbsp;|&nbsp; **Est. time:** 30 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Pre-trained Models & Fine-tuning**

### What you'll do
- Get real predictions from a real model on a labelled set
- Build a confusion matrix and compute precision & recall by hand
- See why accuracy alone can mislead

> **How this lab works (near-real):** these labs use **real Hugging Face Transformers** locally on CPU &mdash; a real pretrained sentiment model, a real tokenizer, and (the headline) a **real fine-tune** you run yourself. Read the **Concept**, fill the real `___` blanks in **Build it** (real model / tokenizer / training-loop calls), **Run it for real** to see the **actual model output** (including the real **before &rarr; after** fine-tune numbers), note **What to notice**, then finish with an open **Your turn**. There is **no auto-grader** &mdash; the goal is real results you can read. The genuine maths (softmax, precision/recall) you still compute **by hand** &mdash; real mechanics, not a stub.

> **Models:** small, CPU-friendly models from the HF hub &mdash; `distilbert-base-uncased-finetuned-sst-2-english` (out-of-the-box sentiment + logits), `distilbert-base-uncased` (tokenizer), `prajjwal1/bert-tiny` (frozen features **and** the model you fine-tune). First use downloads the weights (needs network), then they are cached. An optional hosted comparison uses `ChatGroq` (`GROQ_API_KEY` in `.env`).

**Reference:** [Module 4 slides &mdash; Evaluation](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html) &nbsp;&middot;&nbsp; [All Module 4 labs](./index.html)

In [ ]:
# Setup -- run me first
import os, pathlib
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("USE_TF", "0")                 # these labs are torch-only; skip the TF backend
os.environ.setdefault("TF_CPP_MIN_LOG_LEVEL", "3")   # mute TensorFlow's C++ INFO/WARNING startup noise
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv(usecwd=True), override=True)   # GROQ_API_KEY etc. (optional hosted compare)

WORK = os.path.join(os.environ.get("TEMP") or os.environ.get("TMP") or "/tmp", "biaa-lab-04-08")
os.makedirs(WORK, exist_ok=True)
print("WORK:", WORK)
print("Real Hugging Face models load from the hub on first use (one-time download, then cached).")

## Concept
Accuracy hides *how* a model is wrong. The **confusion matrix** counts TP/FP/FN/TN.
**Precision** = TP / (TP+FP) ("when it says positive, is it right?").
**Recall** = TP / (TP+FN) ("of the real positives, how many did it catch?").
We compute these from the **real** distilbert-SST-2 model's actual predictions on a labelled set that
includes a few genuinely tricky sentences.

## Build it
Implement TP/FP/FN and precision/recall (real mechanics).

In [ ]:
def counts(y_true, y_pred):
    TP = ___   # TODO: predicted 1 AND true 1
    FP = ___   # TODO: predicted 1 BUT true 0
    FN = ___   # TODO: predicted 0 BUT true 1
    return TP, FP, FN

def precision(y_true, y_pred):
    TP, FP, FN = counts(y_true, y_pred)
    return ___   # TODO: TP / (TP + FP) if (TP+FP) else 0.0
def recall(y_true, y_pred):
    TP, FP, FN = counts(y_true, y_pred)
    return ___   # TODO: TP / (TP + FN) if (TP+FN) else 0.0

## Run it for real
Get REAL predictions from the model, then measure.

In [ ]:
try:
    from transformers import pipeline
    from sklearn.metrics import confusion_matrix
    clf = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

    EVAL = [("a brilliant moving masterpiece", 1), ("a boring dreadful waste", 0),
            ("i loved every minute", 1), ("terrible and awful", 0),
            ("not bad at all", 1), ("hardly a masterpiece", 0),
            ("it was fine i guess", 1), ("a complete letdown", 0)]
    y_true = [y for _, y in EVAL]
    y_pred = [1 if clf(t)[0]["label"] == "POSITIVE" else 0 for t, _ in EVAL]

    print("y_true:", y_true)
    print("y_pred:", y_pred, " (REAL model predictions)")
    print("precision:", round(precision(y_true, y_pred), 3), "| recall:", round(recall(y_true, y_pred), 3))
    print("confusion matrix [[TN,FP],[FN,TP]]:")
    print(confusion_matrix(y_true, y_pred))
except Exception as e:
    print("(If a ___ is still unfilled, fill it above. On first run the model downloads")
    print(" from the Hugging Face hub -- that needs network; re-run once it finishes.)")
    print("  reason:", type(e).__name__, "--", e)

## What to notice
- The predictions are **real** model output on deliberately tricky text (negation, hedging). This strong SST-2 model actually handles them all here, so precision and recall come out perfect &mdash; a weaker model (e.g. your fine-tuned bert-tiny) would show real FP/FN. The **framework** is the lesson, whatever the numbers.
- **Precision vs recall** tell you *which kind* of mistake it makes: false positives hurt precision, missed positives hurt recall.
- The **confusion matrix** shows the full breakdown at a glance &mdash; always read it before trusting a single accuracy number.

## Your turn (open task &mdash; no grader)
Add your own hard cases (sarcasm, mixed sentiment) to `EVAL` and re-run. Can you drive
precision and recall apart (make one high, the other low)? On a high-stakes task, which matters more &mdash;
catching every positive (recall) or being right when you flag one (precision)? A "good" answer names a
real task for each.

---
### Key takeaway
On imbalanced or high-stakes tasks, precision and recall matter more than accuracy. Always read the confusion matrix of **real** predictions before you trust a score.

[&#8592; All Module 4 labs](./index.html) &nbsp;&middot;&nbsp; [Module 4 slides](../../presentation/day2-module4-pretrained-models-and-fine-tuning.html) &nbsp;&middot;&nbsp; [Course outline](../../course-outline-building-intelligent-ai-agents.html)

<sub>&copy; 2026 Gheware DevOps &amp; Agentic AI &middot; Building Intelligent AI Agents &middot; devops.gheware.com &middot; Trainer: Rajesh Gheware</sub>